# Vector Autoregression (VAR) Models

A **Vector Autoregression** of order $p$, written VAR($p$), is the
multivariate generalization of the univariate AR($p$) model.  Instead
of a single equation, we have a *system* of equations -- one for each
variable -- where every variable depends on its own lags **and** the
lags of all other variables in the system.

VAR models are the workhorse of applied multivariate time series
analysis.  They are used for:
- Joint forecasting of interrelated series
- Understanding dynamic interactions (Granger causality)
- Tracing how shocks propagate through a system (impulse responses)

This notebook covers:
1. VAR(1) for two variables -- the building block
2. General VAR($p$) in matrix notation
3. Stationarity requirements and differencing
4. Lag order selection with AIC/BIC
5. Fitting VAR models with `statsmodels`
6. Granger causality tests
7. Impulse response functions
8. Forecasting and inverse-differencing
9. Application to real economic data
10. Application to river discharge data

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")
rng = np.random.default_rng(42)

---
## 1. The VAR(1) Model for Two Variables

The simplest multivariate case: two variables $y_{1,t}$ and $y_{2,t}$,
each depending on one lag of both variables.

$$
y_{1,t} = c_1 + \phi_{11} y_{1,t-1} + \phi_{12} y_{2,t-1} + \varepsilon_{1,t}
$$

$$
y_{2,t} = c_2 + \phi_{21} y_{1,t-1} + \phi_{22} y_{2,t-1} + \varepsilon_{2,t}
$$

Key differences from two separate AR(1) models:
- $\phi_{12}$ captures the effect of $y_2$'s past on $y_1$
- $\phi_{21}$ captures the effect of $y_1$'s past on $y_2$
- The error terms $\varepsilon_{1,t}$ and $\varepsilon_{2,t}$ may be contemporaneously correlated

In [ ]:
def simulate_var1(Phi, c, n=400, seed=42):
    """Simulate a bivariate VAR(1) process.

    Parameters
    ----------
    Phi : (2, 2) array — coefficient matrix
    c   : (2,) array — intercept vector
    n   : int — number of observations
    seed : int — random seed

    Returns
    -------
    Y : (n, 2) array
    """
    rng_local = np.random.default_rng(seed)
    K = 2
    Y = np.zeros((n, K))
    for t in range(1, n):
        eps = rng_local.standard_normal(K)
        Y[t] = c + Phi @ Y[t - 1] + eps
    return Y


# Simulate a bivariate VAR(1) with cross-variable effects
Phi = np.array([
    [0.5,  0.3],   # y1 depends positively on both y1_lag and y2_lag
    [0.2,  0.4],   # y2 depends positively on both y1_lag and y2_lag
])
c = np.array([0.0, 0.0])

Y_sim = simulate_var1(Phi, c, n=400)
sim_df = pd.DataFrame(Y_sim, columns=["y1", "y2"])

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(sim_df["y1"], label="$y_1$", color="tab:blue")
axes[0].set_title("Simulated VAR(1) — $y_1$")
axes[0].legend()
axes[1].plot(sim_df["y2"], label="$y_2$", color="tab:orange")
axes[1].set_title("Simulated VAR(1) — $y_2$")
axes[1].legend()
axes[1].set_xlabel("t")
plt.suptitle(r"VAR(1): $\Phi = [[0.5, 0.3], [0.2, 0.4]]$", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("The two series move together — high values of y1 tend to precede")
print("high values of y2 (and vice versa) due to the cross-variable effects.")

In [ ]:
# Cross-correlation between the two simulated series
fig, ax = plt.subplots(figsize=(10, 4))
max_lag = 15
ccf_vals = [sim_df["y1"].corr(sim_df["y2"].shift(k)) for k in range(-max_lag, max_lag + 1)]
lags = range(-max_lag, max_lag + 1)
ax.stem(lags, ccf_vals, linefmt="tab:blue", markerfmt="o", basefmt="grey")
ax.axhline(0, color="grey", linestyle="--", alpha=0.5)
ax.set_xlabel("Lag")
ax.set_ylabel("Cross-correlation")
ax.set_title("Cross-Correlation Function: $y_1$ and $y_2$")
plt.tight_layout()
plt.show()

print("Strong contemporaneous and lagged cross-correlations confirm")
print("the variables are dynamically interrelated.")

---
## 2. General VAR($p$) in Matrix Notation

For $K$ variables observed over time, the VAR($p$) model is:

$$
\mathbf{y}_t = \mathbf{c} + \mathbf{\Phi}_1 \mathbf{y}_{t-1} + \mathbf{\Phi}_2 \mathbf{y}_{t-2} + \cdots + \mathbf{\Phi}_p \mathbf{y}_{t-p} + \boldsymbol{\varepsilon}_t
$$

where:
- $\mathbf{y}_t$ is a $K \times 1$ vector of observations at time $t$
- $\mathbf{c}$ is a $K \times 1$ intercept vector
- $\mathbf{\Phi}_i$ is a $K \times K$ coefficient matrix for lag $i$
- $\boldsymbol{\varepsilon}_t \sim N(\mathbf{0}, \mathbf{\Sigma})$ is a $K \times 1$ white-noise vector

### Number of parameters

A VAR($p$) with $K$ variables has $K + K^2 p$ parameters (intercepts + coefficients).
For $K = 3$ and $p = 4$, that is $3 + 9 \times 4 = 39$ parameters.
This rapid growth is why VAR models are typically limited to a small
number of variables and moderate lag orders.

### Stationarity

The process is stationary if all eigenvalues of the companion matrix
lie inside the unit circle.  In practice, this means:
- Test each series for unit roots (ADF test)
- Difference any non-stationary series before fitting

In [ ]:
# Demonstrate the parameter explosion
param_table = []
for K in [2, 3, 5, 10]:
    for p in [1, 2, 4, 8]:
        n_params = K + K * K * p
        param_table.append({"K (variables)": K, "p (lags)": p, "Parameters": n_params})

param_df = pd.DataFrame(param_table)
print(param_df.pivot(index="p (lags)", columns="K (variables)", values="Parameters"))
print()
print("Lesson: keep K small and p moderate, or you will overfit.")
print("Most applied VAR models use 2-5 variables and 1-12 lags.")

---
## 3. Stationarity Check and Differencing

VAR requires all input series to be stationary.  We use the Augmented
Dickey-Fuller (ADF) test to check each series.  If a series has a unit
root, we difference it before including it in the VAR.

In [ ]:
def adf_summary(series, name=""):
    """Run the ADF test and print a compact summary."""
    result = adfuller(series.dropna(), autolag="AIC")
    stat, pvalue = result[0], result[1]
    conclusion = "Stationary" if pvalue < 0.05 else "Non-stationary"
    print(f"{name:>25s}  ADF stat={stat:+.4f}  p={pvalue:.4f}  => {conclusion}")
    return pvalue < 0.05


# Check stationarity of our simulated data (should be stationary)
print("Simulated VAR(1) data:")
adf_summary(sim_df["y1"], "y1")
adf_summary(sim_df["y2"], "y2")
print("\nBoth stationary — as expected from the simulation design.")

In [ ]:
# Fit VAR to simulated data and recover the true coefficient matrix
model_sim = VAR(sim_df)
result_sim = model_sim.fit(maxlags=5, ic="aic")

print(f"Selected lag order: {result_sim.k_ar}")
print()
print("Estimated Phi_1 matrix:")
print(result_sim.coefs[0].round(3))
print()
print("True Phi_1 matrix:")
print(Phi)
print()
print("The estimates are close to the true values.")

---
## 4. Lag Order Selection

We select the lag order $p$ by fitting VAR($1$), VAR($2$), $\ldots$ and
picking the model with the lowest information criterion:

- **AIC** (Akaike) — tends to select more lags (better for forecasting)
- **BIC** (Bayesian/Schwarz) — tends to select fewer lags (more parsimonious)
- **HQIC** (Hannan-Quinn) — a middle ground

The `VAR` class in statsmodels provides `model.select_order(maxlags)` to
compare all criteria at once.

In [ ]:
# Lag order selection on simulated data
model_sim = VAR(sim_df)
lag_selection = model_sim.select_order(maxlags=10)
print(lag_selection.summary())
print()
print("All criteria should point to p=1 (the true order).")

---
## 5. Application to Real Data: Money Supply and Consumer Spending

We now apply VAR to real macroeconomic data:
- **M2 Money Stock** (M2SLMoneyStock.csv) — a broad measure of money supply
- **Personal Consumption Expenditures** (PCEPersonalSpending.csv) — consumer spending

Economic theory suggests these variables are related: money supply affects
spending power, and consumer activity influences monetary policy.

In [ ]:
# Load and prepare the economic data
m2 = pd.read_csv(
    DATA_DIR / "M2SLMoneyStock.csv",
    index_col="Date",
    parse_dates=True,
)
pce = pd.read_csv(
    DATA_DIR / "PCEPersonalSpending.csv",
    index_col="Date",
    parse_dates=True,
)

# Combine into a single DataFrame
econ = pd.concat([m2, pce], axis=1).dropna()
econ.columns = ["M2", "PCE"]
econ.index.freq = pd.infer_freq(econ.index)

print(f"Shape: {econ.shape}")
print(f"Date range: {econ.index[0].date()} to {econ.index[-1].date()}")
print(f"Frequency: {econ.index.freq}")
econ.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(econ["M2"], label="M2 Money Stock", color="tab:blue")
axes[0].set_ylabel("Billions $")
axes[0].set_title("M2 Money Stock")
axes[0].legend()

axes[1].plot(econ["PCE"], label="Personal Consumption Expenditures", color="tab:orange")
axes[1].set_ylabel("Billions $")
axes[1].set_title("Personal Consumption Expenditures (PCE)")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Both series have strong upward trends — clearly non-stationary.")
print("We must difference them before fitting a VAR.")

In [ ]:
# ADF tests on levels
print("ADF tests on LEVELS:")
adf_summary(econ["M2"], "M2 (level)")
adf_summary(econ["PCE"], "PCE (level)")

print()

# First-difference the series
econ_diff = econ.diff().dropna()
econ_diff.columns = ["dM2", "dPCE"]

print("ADF tests on FIRST DIFFERENCES:")
adf_summary(econ_diff["dM2"], "dM2 (differenced)")
adf_summary(econ_diff["dPCE"], "dPCE (differenced)")

print()
print("After first-differencing, both series are stationary.")

In [ ]:
# Plot differenced series
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(econ_diff["dM2"], color="tab:blue", linewidth=0.8)
axes[0].set_title("First Difference: $\\Delta$ M2 Money Stock")
axes[0].axhline(0, color="grey", linestyle="--", alpha=0.5)

axes[1].plot(econ_diff["dPCE"], color="tab:orange", linewidth=0.8)
axes[1].set_title("First Difference: $\\Delta$ PCE")
axes[1].axhline(0, color="grey", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

print("Differenced series fluctuate around zero — suitable for VAR.")

In [ ]:
# Train/test split (hold out last 24 months)
n_test = 24
train_econ = econ_diff.iloc[:-n_test]
test_econ = econ_diff.iloc[-n_test:]

print(f"Train: {len(train_econ)} months ({train_econ.index[0].date()} to {train_econ.index[-1].date()})")
print(f"Test:  {len(test_econ)} months ({test_econ.index[0].date()} to {test_econ.index[-1].date()})")

In [ ]:
# Lag order selection
model_econ = VAR(train_econ)
lag_order = model_econ.select_order(maxlags=15)
print(lag_order.summary())

selected_p = lag_order.aic
print(f"\nSelected lag order (AIC): p = {selected_p}")

In [ ]:
# Fit VAR with selected lag order
result_econ = model_econ.fit(maxlags=15, ic="aic")
print(result_econ.summary())

---
## 6. Granger Causality

**Granger causality** asks: does the past of variable $x$ contain
information useful for predicting $y$, beyond what $y$'s own past
provides?

Formally, $x$ Granger-causes $y$ if the coefficients on lagged $x$
in the equation for $y$ are jointly significantly different from zero.

Important caveats:
- Granger causality is about **predictive precedence**, not true causation
- It is sensitive to the number of lags included
- Omitted variables can create spurious Granger causality

In [ ]:
# Granger causality within the fitted VAR
# Test: does dPCE Granger-cause dM2?
gc_m2 = result_econ.test_causality("dM2", causing="dPCE", kind="f")
print("H0: dPCE does NOT Granger-cause dM2")
print(f"  F-statistic: {gc_m2.test_statistic:.4f}")
print(f"  p-value:     {gc_m2.pvalue:.4f}")
print(f"  Conclusion:  {'Reject H0 — dPCE Granger-causes dM2' if gc_m2.pvalue < 0.05 else 'Fail to reject H0'}")

print()

# Test: does dM2 Granger-cause dPCE?
gc_pce = result_econ.test_causality("dPCE", causing="dM2", kind="f")
print("H0: dM2 does NOT Granger-cause dPCE")
print(f"  F-statistic: {gc_pce.test_statistic:.4f}")
print(f"  p-value:     {gc_pce.pvalue:.4f}")
print(f"  Conclusion:  {'Reject H0 — dM2 Granger-causes dPCE' if gc_pce.pvalue < 0.05 else 'Fail to reject H0'}")

In [ ]:
# Alternative: use statsmodels.tsa.stattools.grangercausalitytests
# This tests Granger causality at multiple lags and reports F-test p-values
print("Granger causality: dM2 -> dPCE (does M2 help predict PCE?)")
print("=" * 60)
gc_results = grangercausalitytests(
    train_econ[["dPCE", "dM2"]],  # note: column order matters — [y, x]
    maxlag=6,
    verbose=True,
)

---
## 7. Impulse Response Functions (IRF)

An **impulse response function** traces the effect of a one-standard-deviation
shock to one variable on all variables in the system over subsequent periods.

For example, if we shock money supply ($\varepsilon_{M2}$), the IRF shows:
- How M2 itself responds over time (should decay back to zero if stationary)
- How PCE responds to the M2 shock over time

IRFs are one of the most useful outputs of VAR models for understanding
dynamic relationships.

In [ ]:
# Compute and plot impulse response functions
irf = result_econ.irf(periods=24)
irf.plot(orth=False)
plt.suptitle("Impulse Response Functions (24 periods)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Each subplot shows how a 1-unit shock to the 'impulse' variable")
print("affects the 'response' variable over the next 24 months.")
print("Responses should decay to zero for a stationary system.")

In [ ]:
# Plot cumulative impulse response (useful for differenced data)
irf.plot_cum(orth=False)
plt.suptitle("Cumulative Impulse Response Functions", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Cumulative IRFs show the total accumulated effect of the shock.")
print("Since we are working with differenced data, the cumulative IRF")
print("corresponds to the effect on the level of the original series.")

---
## 8. Forecasting and Inverse-Differencing

The VAR forecast is on the *differenced* data.  To get forecasts on
the original scale, we must **inverse-difference** — that is, cumulatively
add the predicted differences back to the last observed level.

If $\hat{\Delta y}_{T+1}, \hat{\Delta y}_{T+2}, \ldots$ are the forecasted
differences, then the level forecast is:

$$
\hat{y}_{T+h} = y_T + \sum_{i=1}^{h} \hat{\Delta y}_{T+i}
$$

In [ ]:
# Forecast the differenced series
k_ar = result_econ.k_ar
lag_data = train_econ.values[-k_ar:]
forecast_diff = result_econ.forecast(lag_data, steps=n_test)

forecast_diff_df = pd.DataFrame(
    forecast_diff,
    index=test_econ.index,
    columns=["dM2_forecast", "dPCE_forecast"],
)

print("Forecast (differenced scale):")
forecast_diff_df.head()

In [ ]:
# Inverse-difference: convert forecasted differences back to levels
# Last known level values (before the test period)
last_m2_level = econ["M2"].iloc[-(n_test + 1)]
last_pce_level = econ["PCE"].iloc[-(n_test + 1)]

forecast_m2_level = last_m2_level + forecast_diff_df["dM2_forecast"].cumsum()
forecast_pce_level = last_pce_level + forecast_diff_df["dPCE_forecast"].cumsum()

# Actual levels in the test period
actual_m2_level = econ["M2"].iloc[-n_test:]
actual_pce_level = econ["PCE"].iloc[-n_test:]

print(f"Last known M2 level:  {last_m2_level:.1f}")
print(f"Last known PCE level: {last_pce_level:.1f}")

In [ ]:
# Plot level forecasts vs actuals
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# M2
axes[0].plot(actual_m2_level, label="Actual M2", color="tab:blue")
axes[0].plot(forecast_m2_level, label="VAR Forecast", color="tab:red", linestyle="--")
axes[0].set_title("M2 Money Stock — Level Forecast")
axes[0].set_ylabel("Billions $")
axes[0].legend()

# PCE
axes[1].plot(actual_pce_level, label="Actual PCE", color="tab:orange")
axes[1].plot(forecast_pce_level, label="VAR Forecast", color="tab:red", linestyle="--")
axes[1].set_title("PCE — Level Forecast")
axes[1].set_ylabel("Billions $")
axes[1].legend()

plt.suptitle("VAR Forecast (inverse-differenced to levels)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate forecast accuracy on differenced scale
from sklearn.metrics import mean_absolute_error, mean_squared_error

for col, fcol in [("dM2", "dM2_forecast"), ("dPCE", "dPCE_forecast")]:
    actual = test_econ[col].values
    pred = forecast_diff_df[fcol].values
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    print(f"{col}:  MAE={mae:.2f}  RMSE={rmse:.2f}")

In [ ]:
# Evaluate on level scale
for name, actual, pred in [
    ("M2", actual_m2_level, forecast_m2_level),
    ("PCE", actual_pce_level, forecast_pce_level),
]:
    mae = mean_absolute_error(actual.values, pred.values)
    rmse = np.sqrt(mean_squared_error(actual.values, pred.values))
    mape = np.mean(np.abs((actual.values - pred.values) / actual.values)) * 100
    print(f"{name} (levels):  MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.2f}%")

---
## 9. Residual Diagnostics

In [ ]:
# Check residuals of the fitted VAR
resid = result_econ.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# dM2 residuals
axes[0, 0].plot(resid["dM2"], linewidth=0.6)
axes[0, 0].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[0, 0].set_title("Residuals: dM2 equation")

plot_acf(resid["dM2"], lags=20, ax=axes[0, 1], title="ACF of dM2 Residuals")

# dPCE residuals
axes[1, 0].plot(resid["dPCE"], linewidth=0.6)
axes[1, 0].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[1, 0].set_title("Residuals: dPCE equation")

plot_acf(resid["dPCE"], lags=20, ax=axes[1, 1], title="ACF of dPCE Residuals")

plt.tight_layout()
plt.show()

print("Good residuals should look like white noise: no autocorrelation,")
print("centered at zero, roughly constant variance.")

In [ ]:
# Durbin-Watson test for autocorrelation in residuals
from statsmodels.stats.stattools import durbin_watson

dw = durbin_watson(resid)
for col, dw_stat in zip(resid.columns, dw):
    print(f"Durbin-Watson ({col}): {dw_stat:.4f}")

print()
print("Values near 2.0 indicate no significant autocorrelation in residuals.")

---
## 10. Application to River Discharge Data

Let us apply VAR to a different domain: annual river discharge levels for
the Rhine and Danube rivers.  These rivers share a common central European
climate, so their discharge levels may be dynamically interrelated.

In [ ]:
# Load river discharge data
rhine = pd.read_csv(DATA_DIR / "rhine_river_discharge.csv", index_col="Year")
danube = pd.read_csv(DATA_DIR / "danube_river_discharge.csv", index_col="Year")

rivers = pd.concat([rhine, danube], axis=1).dropna()
rivers.columns = ["Rhine", "Danube"]

print(f"Shape: {rivers.shape}")
print(f"Year range: {rivers.index[0]} to {rivers.index[-1]}")
rivers.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(rivers.index, rivers["Rhine"], label="Rhine", alpha=0.8)
ax.plot(rivers.index, rivers["Danube"], label="Danube", alpha=0.8)
ax.set_xlabel("Year")
ax.set_ylabel("Discharge (m$^3$/s)")
ax.set_title("Annual River Discharge: Rhine and Danube")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Correlation: {rivers['Rhine'].corr(rivers['Danube']):.3f}")

In [ ]:
# Check stationarity
print("ADF tests on river discharge levels:")
rhine_stationary = adf_summary(rivers["Rhine"], "Rhine")
danube_stationary = adf_summary(rivers["Danube"], "Danube")

if not (rhine_stationary and danube_stationary):
    print("\nDifferencing if needed...")
    rivers_model = rivers.diff().dropna()
    rivers_model.columns = ["dRhine", "dDanube"]
    print("\nADF on differenced:")
    adf_summary(rivers_model["dRhine"], "dRhine")
    adf_summary(rivers_model["dDanube"], "dDanube")
    differenced = True
else:
    rivers_model = rivers.copy()
    differenced = False
    print("\nBoth series appear stationary — can fit VAR on levels.")

In [ ]:
# Fit VAR to river data
model_rivers = VAR(rivers_model)
river_lag_order = model_rivers.select_order(maxlags=10)
print(river_lag_order.summary())

result_rivers = model_rivers.fit(maxlags=10, ic="aic")
print(f"\nSelected order: p = {result_rivers.k_ar}")

In [ ]:
# Granger causality for river data
cols = rivers_model.columns.tolist()

gc1 = result_rivers.test_causality(cols[0], causing=cols[1], kind="f")
print(f"Does {cols[1]} Granger-cause {cols[0]}?")
print(f"  F={gc1.test_statistic:.4f}, p={gc1.pvalue:.4f}")
print()

gc2 = result_rivers.test_causality(cols[1], causing=cols[0], kind="f")
print(f"Does {cols[0]} Granger-cause {cols[1]}?")
print(f"  F={gc2.test_statistic:.4f}, p={gc2.pvalue:.4f}")

In [ ]:
# Impulse response functions for river data
irf_rivers = result_rivers.irf(periods=15)
irf_rivers.plot(orth=False)
plt.suptitle("IRF — River Discharge", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("A shock to Rhine discharge may propagate to Danube discharge")
print("(or vice versa) if they share common climate drivers.")

---
## Summary

- **VAR($p$)** models $K$ interrelated time series jointly, where each
  variable depends on $p$ lags of *all* variables:
  $\mathbf{y}_t = \mathbf{c} + \mathbf{\Phi}_1 \mathbf{y}_{t-1} + \cdots + \mathbf{\Phi}_p \mathbf{y}_{t-p} + \boldsymbol{\varepsilon}_t$

- **Stationarity** is required: test with ADF, difference if needed

- **Lag order**: use `model.select_order(maxlags)` to compare AIC/BIC

- **Granger causality**: `results.test_causality()` tests whether one
  variable's history helps predict another

- **Impulse response functions**: `results.irf(periods)` traces how shocks
  propagate through the system

- **Inverse-differencing**: if you differenced before fitting, you must
  cumsum the forecasted differences and add back the last known level

- **Parameter growth** ($K + K^2 p$) limits VAR to a modest number of
  variables — typically 2-5 in practice

In [ ]:
print("Next notebook: VARMA and VECM — extending VAR with MA terms and cointegration.")